# **Validación de pyR0compute con modelos publicados**

Este notebook reproduce la Tabla de validación del artículo de pyR0compute. Cada modelo se escribe tal como aparece en la fuente, se calcula $\mathcal{R}_0$ con `R0Model` y se compara con la expresión publicada.

**Fuentes**

* van den Driessche, P., Watmough, J. (2002). Reproduction numbers and sub-threshold endemic equilibria for compartmental models of disease transmission. *Math. Biosci.* 180, 29-48. https://doi.org/10.1016/S0025-5564(02)00108-6
* Ciupe, S.M., Heffernan, J.M. (2017). In-host modeling. *Infect. Dis. Model.* 2, 188-202. https://doi.org/10.1016/j.idm.2017.04.002
* Cuesta-Herrera, L., Pastenes, L., Arencibia, A.D., Córdova-Lepe, F., Montoya, C. (2024). Dynamics of activation and regulation of the immune response to attack by viral pathogens using mathematical modeling. *Mathematics* 12, 2681. https://doi.org/10.3390/math12172681
* Cuesta-Herrera, L., Pastenes, L., Córdova-Lepe, F., Arencibia, A.D. (2025). Mathematical modeling of the immune response mediated by human T-helper lymphocytes in viral diseases. *Math. Biosci. Eng.* 22(11), 2807-2825. https://doi.org/10.3934/mbe.2025103

**Criterio de coincidencia**

| Caso | Criterio |
|---|---|
| General | la diferencia simbólica entre el $\mathcal{R}_0$ obtenido y el publicado simplifica a 0 |
| Resultados con raíz cuadrada | se comparan los cuadrados |
| Modelo multicepa | se compara el conjunto de valores propios no nulos de $FV^{-1}$ |
| Equilibrio con raíz cuadrada (Ciupe y Heffernan, Ec. 3) | forma compacta y valor numérico en 36 conjuntos de parámetros |
| Modelo de 7 compartimentos (Cuesta-Herrera et al., 2024) | forma compacta, sustituyendo los valores del equilibrio que usa la fuente |

Los tiempos son la mediana de tres ejecuciones de `R0Model` y `R0_compact`. La tabla final se guarda en `outputs/07_validacion.csv`.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yvan-BM-Git/pyR0compute/blob/main/examples/spanish/07_validacion.ipynb)

In [1]:
%pip install pyR0compute                      # Instala la librería pyR0compute desde PyPI

In [2]:
import os                                     # Carpeta para guardar la tabla
import time                                   # Medición de tiempos
import platform                               # Datos del equipo
import statistics                             # Mediana de los tiempos
import warnings                               # Avisos de la librería
import numpy as np                            # Cálculo numérico
import pandas as pd                           # Tabla de resultados
import sympy as sp                            # Cálculo simbólico
import pyr0compute                            # Versión de la librería
from pyr0compute import R0Model               # Clase principal de la librería
from pyr0compute import parse_expression as P # Texto -> expresión de SymPy

os.makedirs("outputs", exist_ok=True)         # La tabla se guarda en outputs/
warnings.simplefilter("ignore")               # Los avisos se discuten en el texto
print("pyR0compute", pyr0compute.__version__, "| SymPy", sp.__version__,
      "| Python", platform.python_version(), "|", platform.processor() or platform.machine())

resultados = []                               # Una fila por caso de validación


def mismo(a, b):
    """True si a - b simplifica a 0."""
    return sp.simplify(sp.sympify(a) - sp.sympify(b)) == 0


def validar(caso, fuente, texto, infectados, publicado, comparar=None, **opciones):
    """Calcula R0, lo compara con el publicado y registra tiempo y supuestos."""
    tiempos = []
    for _ in range(3):                        # Mediana de tres ejecuciones
        t0 = time.perf_counter()
        modelo = R0Model(texto, infected=infectados, **opciones)
        modelo.R0_compact
        tiempos.append(time.perf_counter() - t0)
    coincide = comparar(modelo) if comparar else mismo(modelo.R0_compact, publicado)
    supuestos = modelo.check_assumptions()    # (A1)-(A5) de van den Driessche y Watmough
    resultados.append({
        "caso": caso, "fuente": fuente,
        "n": len(modelo.variables), "m": len(infectados),
        "coincide": bool(coincide),
        "t_s": round(statistics.median(tiempos), 2),
        **{a: supuestos[a].status for a in ("A1", "A2", "A3", "A4", "A5")},
    })
    print(f"{caso}: coincide = {coincide}, t = {statistics.median(tiempos):.2f} s")
    display(modelo.R0_compact)
    return modelo

pyR0compute 0.3.1 | SymPy 1.14.0 | Python 3.13.16 | x86_64


## 1. Modelos epidemiológicos de van den Driessche y Watmough (2002)

### 1.1 Modelo con tratamiento (sección 4.1)

Compartimentos infectados $E$ e $I$. El fallo del tratamiento $pr_2I$ es una transición, no una nueva infección. La fuente obtiene
$$\mathcal{R}_0=\frac{\beta_1\nu}{(d+\nu+r_1)(d+r_2)-\nu p r_2}.$$
Su Tabla 1 muestra dos descomposiciones alternativas: (a) el fallo del tratamiento como nueva infección y (b) además la progresión $\nu E$. Se declaran con `new_infections`.

In [3]:
TRATAMIENTO = """
    N = S + E + I + T
    dE/dt = beta1*S*I/N + beta2*T*I/N - (d + nu + r1)*E + p*r2*I
    dI/dt = nu*E - (d + r2)*I
    dS/dt = b - d*S - beta1*S*I/N
    dT/dt = -d*T + r1*E + q*r2*I - beta2*T*I/N
"""
validar("Tratamiento", "vdW 2002, 4.1", TRATAMIENTO, ["E", "I"],
        P("beta1*nu/((d + nu + r1)*(d + r2) - nu*p*r2)"))

validar("Tratamiento, Tabla 1(a)", "vdW 2002, 4.1", TRATAMIENTO, ["E", "I"],
        P("nu*(beta1 + p*r2)/((d + nu + r1)*(d + r2))"),
        new_infections={"E": "beta1*S*I/N + beta2*T*I/N + p*r2*I", "I": "0"})

validar("Tratamiento, Tabla 1(b)", "vdW 2002, 4.1", TRATAMIENTO, ["E", "I"],
        None,                                 # Resultado con raíz: se comparan cuadrados
        comparar=lambda m: mismo(m.R0**2, P("nu*(beta1 + p*r2)/((d + nu + r1)*(d + r2))")),
        new_infections={"E": "beta1*S*I/N + beta2*T*I/N + p*r2*I", "I": "nu*E"});

Tratamiento: coincide = True, t = 0.15 s


-beta1*nu/(-d**2 - d*nu - d*r1 - d*r2 + nu*p*r2 - nu*r2 - r1*r2)

Tratamiento, Tabla 1(a): coincide = True, t = 0.15 s


nu*(beta1 + p*r2)/((d + r2)*(d + nu + r1))

Tratamiento, Tabla 1(b): coincide = True, t = 0.16 s


sqrt(nu*(beta1 + p*r2))/sqrt((d + r2)*(d + nu + r1))

### 1.2 Progresión por etapas (sección 4.3), con cuatro etapas

La última etapa no transmite. La fuente obtiene (Ec. 13)
$$\mathcal{R}_0=\frac{\beta_1}{\nu_1+d_1}+\frac{\beta_2\nu_1}{(\nu_1+d_1)(\nu_2+d_2)}+\frac{\beta_3\nu_1\nu_2}{(\nu_1+d_1)(\nu_2+d_2)(\nu_3+d_3)}.$$

In [4]:
validar("Progresión por etapas", "vdW 2002, 4.3", """
    N = S + I1 + I2 + I3 + I4
    dI1/dt = (beta1*I1 + beta2*I2 + beta3*I3)*S/N - (v1 + d1)*I1
    dI2/dt = v1*I1 - (v2 + d2)*I2
    dI3/dt = v2*I2 - (v3 + d3)*I3
    dI4/dt = v3*I3 - d4*I4
    dS/dt = b - b*S - (beta1*I1 + beta2*I2 + beta3*I3)*S/N
""", ["I1", "I2", "I3", "I4"],
    P("beta1/(v1 + d1) + beta2*v1/((v1 + d1)*(v2 + d2)) + beta3*v1*v2/((v1 + d1)*(v2 + d2)*(v3 + d3))"));

Progresión por etapas: coincide = True, t = 0.27 s


(beta1*d2*d3 + beta1*d2*v3 + beta1*d3*v2 + beta1*v2*v3 + beta2*d3*v1 + beta2*v1*v3 + beta3*v1*v2)/((d1 + v1)*(d2 + v2)*(d3 + v3))

### 1.3 Multicepa con superinfección (sección 4.4)

$FV^{-1}$ tiene dos valores propios, $\beta_i/(b+\gamma_i)$, y $\mathcal{R}_0$ es el mayor (Ecs. 16 y 17). El término $\nu I_1 I_2$ es una transferencia entre infectados.

In [5]:
b, g1, g2, b1, b2 = sp.symbols("b gamma1 gamma2 beta1 beta2")
validar("Multicepa", "vdW 2002, 4.4", """
    dI1/dt = beta1*I1*S - (b + gamma1)*I1 + nu*I1*I2
    dI2/dt = beta2*I2*S - (b + gamma2)*I2 - nu*I1*I2
    dS/dt = b - b*S + gamma1*I1 + gamma2*I2 - (beta1*I1 + beta2*I2)*S
""", ["I1", "I2"], None,
    comparar=lambda m: {sp.simplify(e) for e in m.eigenvalues} == {b1/(b + g1), b2/(b + g2)});

Multicepa: coincide = True, t = 0.12 s


Max(beta1/(b + gamma1), beta2/(b + gamma2))

### 1.4 Vector-huésped (sección 4.5)

La fuente obtiene (Ec. 21) $\mathcal{R}_0=\sqrt{\beta_s\beta_m/(c(b+\gamma))}$; se comparan los cuadrados.

In [6]:
validar("Vector-huésped", "vdW 2002, 4.5", """
    dI/dt = beta_s*S*V - (b + gamma)*I
    dV/dt = beta_m*M*I - c*V
    dS/dt = b - b*S + gamma*I - beta_s*S*V
    dM/dt = c - c*M - beta_m*M*I
""", ["I", "V"], None,
    comparar=lambda m: mismo(m.R0**2, P("beta_s*beta_m/(c*(b + gamma))")));

Vector-huésped: coincide = True, t = 0.08 s


sqrt(beta_m*beta_s)/sqrt(c*(b + gamma))

## 2. Modelos intra-huésped de Ciupe y Heffernan (2017)

### 2.1 Infección crónica, Ec. (1)

$\mathcal{R}_0 = p\beta T_0/(\delta c)$ con $T_0 = s/d$ (Ec. 2).

In [7]:
validar("Infección crónica", "CH 2017, Ec. (1)", """
    dT/dt = s - d*T - beta*T*V
    dI/dt = beta*T*V - delta*I
    dV/dt = p*I - c*V
""", ["I", "V"], P("p*beta*(s/d)/(delta*c)"));

Infección crónica: coincide = True, t = 0.04 s


beta*p*s/(c*d*delta)

### 2.2 Proliferación hepática y cura, Ec. (3)

El equilibrio $T_0$ es la raíz positiva de una cuadrática, y la fuente obtiene (Ec. 4)
$$\mathcal{R}_0=\frac{p\beta T_0}{c\left[\rho+\delta+r_I\left(T_0/T_m-1\right)\right]}.$$
La librería entrega $\mathcal{R}_0$ en términos de $T^{*}$; se compara la forma compacta con $T^{*}=T_0$ y además el valor numérico en 36 conjuntos de parámetros con denominador positivo.

In [8]:
def comparar_ec3(m):
    T_star = next(s for s in m.R0_compact.free_symbols if s.name == "T_star")
    s_, rT, rI, Tm, d, beta, p, c, delta, rho = sp.symbols("s r_T r_I T_m d beta p c delta rho")
    compacto = m.R0_compact.xreplace({x: sp.Symbol(x.name) for x in m.R0_compact.free_symbols})
    forma = mismo(compacto, p*beta*sp.Symbol("T_star")/(c*(rho + delta + rI*(sp.Symbol("T_star")/Tm - 1))))
    rng = np.random.default_rng(1)
    ok, n = True, 0
    while n < 36:
        v = {k: float(10**rng.uniform(-2, 1)) for k in "s r_T r_I T_m d beta p c delta rho".split()}
        T0 = v["T_m"]/(2*v["r_T"])*(v["r_T"] - v["d"] + np.sqrt((v["r_T"] - v["d"])**2 + 4*v["s"]*v["r_T"]/v["T_m"]))
        den = v["rho"] + v["delta"] + v["r_I"]*(T0/v["T_m"] - 1)
        if den <= 0:
            continue                          # Fuera del rango donde V es una M-matriz
        publicado = v["p"]*v["beta"]*T0/(v["c"]*den)
        ok &= abs(m.R0_numeric(v) - publicado) <= 1e-9*publicado
        n += 1
    print("forma compacta:", forma, "| 36 puntos numéricos:", ok)
    return forma and ok

ec3 = validar("Proliferación hepática", "CH 2017, Ec. (3)", """
    dT/dt = s + r_T*T*(1 - (T + I)/T_m) - d*T - beta*T*V + rho*I
    dI/dt = beta*T*V + r_I*I*(1 - (T + I)/T_m) - delta*I - rho*I
    dV/dt = p*I - c*V
""", ["I", "V"], None, comparar=comparar_ec3)
ec3.dfe_definitions                           # T* en forma cerrada

forma compacta: True | 36 puntos numéricos: True


Proliferación hepática: coincide = True, t = 0.23 s


T_star*T_m*beta*p/(c*(T_star*r_I + T_m*delta - T_m*r_I + T_m*rho))

{T_star: (sqrt(T_m)*sqrt(T_m*d**2 - 2*T_m*d*r_T + T_m*r_T**2 + 4*r_T*s) - T_m*(d - r_T))/(2*r_T)}

### 2.3 Infección aguda con fase latente, Ec. (6)

Sin producción ni muerte de células diana, todo estado $(N,0,0,0)$ es un equilibrio libre de enfermedad: el equilibrio se declara con `dfe={"T": "N"}`. La fuente indica que $\mathcal{R}_0$ es el de la Ec. (2) con $T_0=N$.

In [9]:
validar("Infección aguda con latencia", "CH 2017, Ec. (6)", """
    dT/dt = -beta*T*V
    dE/dt = beta*T*V - k*E
    dI/dt = k*E - delta*I
    dV/dt = p*I - c*V
""", ["E", "I", "V"], P("p*beta*N/(delta*c)"), dfe={"T": "N"});

Infección aguda con latencia: coincide = True, t = 0.04 s


N*beta*p/(c*delta)

### 2.4 Patógeno autorreplicante, Ec. (7) con $f(V)=rV(1-V/K)$

La fuente da dos números (Ecs. 12 y 13): $\mathcal{R}_{0,I}=\dfrac{p}{\delta}\dfrac{\beta T_0}{c-r}$, si la infección empieza con una célula infectada, y $\mathcal{R}_{0,V}=\dfrac{r}{c}+\dfrac{p\beta T_0}{c\delta}$, si empieza con una partícula. La regla automática deja el crecimiento logístico en $\mathcal{V}$ y produce $\mathcal{R}_{0,I}$; declarado como nueva infección, produce $\mathcal{R}_{0,V}$.

In [10]:
AUTORREPLICACION = """
    dT/dt = s - d*T - beta*T*V
    dI/dt = beta*T*V - delta*I
    dV/dt = r*V*(1 - V/K) + p*I - c*V
"""
validar("Autorreplicación, R0,I", "CH 2017, Ec. (7)", AUTORREPLICACION, ["I", "V"],
        P("p/delta*beta*(s/d)/(c - r)"))
validar("Autorreplicación, R0,V", "CH 2017, Ec. (7)", AUTORREPLICACION, ["I", "V"],
        P("r/c + p*beta*(s/d)/(c*delta)"),
        new_infections={"I": "beta*T*V", "V": "r*V*(1 - V/K)"});

Autorreplicación, R0,I: coincide = True, t = 0.11 s


-beta*p*s/(d*delta*(-c + r))

Autorreplicación, R0,V: coincide = True, t = 0.05 s


(beta*p*s + d*delta*r)/(c*d*delta)

### 2.5 Pérdida de patógeno por infección, Ec. (14)

$\mathcal{R}_{0,loss}=\dfrac{\beta T_0}{c+\beta T_0}\dfrac{p}{\delta}$ (Ec. 16).

In [11]:
validar("Pérdida de patógeno", "CH 2017, Ec. (14)", """
    dT/dt = s - d*T - beta*T*V
    dI/dt = beta*T*V - delta*I
    dV/dt = p*I - c*V - beta*T*V
""", ["I", "V"], P("beta*(s/d)*p/((c + beta*s/d)*delta)"));

Pérdida de patógeno: coincide = True, t = 0.13 s


beta*p*s/(delta*(beta*s + c*d))

## 3. Modelos de respuesta inmune de Cuesta-Herrera et al.

### 3.1 Virus sin linfocitos T y con linfocitos T colaboradores (2025, sistemas 2.1 y 2.2)

$\mathcal{R}_0=\kappa(\lambda/d)(\nu/\mu)/\alpha$ y $\mathcal{R}_0^{*}=\dfrac{\kappa\lambda\nu/d}{(\alpha+\beta b/c)(\mu+\tau b/c)}$.

In [12]:
validar("Virus sin linfocitos T", "CH-2025, (2.1)", """
    dE/dt = lambda - d*E - kappa*E*V
    dI/dt = kappa*E*V - alpha*I
    dV/dt = nu*I - mu*V
""", ["I", "V"], P("kappa*(lambda/d)*(nu/mu)/alpha"))

validar("Linfocitos T colaboradores", "CH-2025, (2.2)", """
    dE/dt = lambda - d*E - kappa*E*V
    dI/dt = kappa*E*V - alpha*I - beta*Th*I
    dV/dt = nu*I - mu*V - tau*Th*V
    dTh/dt = b - c*Th + gamma*I*Th
""", ["I", "V"], P("(kappa*lambda*nu/d)/((alpha + beta*b/c)*(mu + tau*b/c))"));

Virus sin linfocitos T: coincide = True, t = 0.07 s


kappa*lambda*nu/(alpha*d*mu)

Linfocitos T colaboradores: coincide = True, t = 0.17 s


c**2*kappa*lambda*nu/(d*(alpha*c + b*beta)*(b*tau + c*mu))

### 3.2 Linfocitos T reguladores (2024, sistema 3)

Siete compartimentos. El equilibrio se resuelve en cuatro bloques, $[E]\to[H,R]\to[C]\to[B]$, y $\mathcal{R}_0$ queda en términos de $B^{*}$ y $C^{*}$. La Ec. (4) de la fuente se recupera al sustituir $C^{*}=g_C/d_C$ y $B^{*}=g_B/d_B$, que son los valores del equilibrio cuando $\tau_C=\rho_C=\tau_B=\rho_B=0$.

In [13]:
PUBLICADO_2024 = P("g_E*(g_B + d_B*z_BV)*(g_C + d_C*z_CI)*nu*tau_E"
                   "/(d_E*(d_I*(g_C + d_C*z_CI) + g_C*tau_I)*(d_V*(g_B + d_B*z_BV) + g_B*tau_V)*z_VE)")

def comparar_2024(m):
    R = m.R0_compact
    star = {s.name: s for s in R.free_symbols}
    sustituido = R.subs({star["C_star"]: P("g_C/d_C"), star["B_star"]: P("g_B/d_B")})
    nombres = {s.name: s for s in PUBLICADO_2024.free_symbols}
    sustituido = sustituido.xreplace({s: nombres.get(s.name, s) for s in sustituido.free_symbols})
    return mismo(sustituido, PUBLICADO_2024)

ch24 = validar("Linfocitos T reguladores", "CH-2024, (3)", """
    dE/dt = g_E - (d_E + tau_E*V/(V + z_VE))*E
    dI/dt = tau_E*V*E/(V + z_VE) - (d_I + tau_I*C/(C + z_CI))*I
    dV/dt = nu*I - (d_V + tau_V*B/(B + z_BV))*V
    dH/dt = g_H - (d_H - tau_H*I/(I + z_IH) + rho_H*R/(R + z_RH))*H
    dC/dt = g_C - (d_C - tau_C*H/(H + z_HC) + rho_C*R/(R + z_RC))*C
    dB/dt = g_B - (d_B - tau_B*H/(H + z_HB) + rho_B*R/(R + z_RB))*B
    dR/dt = g_R - (d_R - tau_R*H/(H + z_HR) + rho_R)*R
""", ["I", "V"], None, comparar=comparar_2024)
for bloque in ch24.dfe_blocks:                # Bloques del equilibrio y tiempo de cada uno
    print(bloque["variables"], bloque["status"], f'{bloque["seconds"]:.2f} s')

Linfocitos T reguladores: coincide = True, t = 7.96 s


g_E*nu*tau_E*(B_star + z_BV)*(C_star + z_CI)/(d_E*z_VE*(B_star*d_V + B_star*tau_V + d_V*z_BV)*(C_star*d_I + C_star*tau_I + d_I*z_CI))

['E'] closed form 0.00 s
['H', 'R'] closed form 6.20 s
['C'] closed form 0.01 s
['B'] closed form 0.01 s


## 4. Tabla de validación

`coincide` indica si el $\mathcal{R}_0$ obtenido coincide con el publicado según el criterio de cada caso. Las columnas A1 a A5 son el resultado de `check_assumptions()`: `holds` (se cumple), `fails` (no se cumple) o `undecided` (no se pudo decidir sin valores de los parámetros).

In [14]:
tabla = pd.DataFrame(resultados)
tabla.to_csv("outputs/07_validacion.csv", index=False)
print(f"{tabla.coincide.sum()} de {len(tabla)} casos coinciden con la fuente")
tabla

15 de 15 casos coinciden con la fuente


,caso,fuente,n,m,coincide,t_s,A1,A2,A3,A4,A5
0,Tratamiento,"vdW 2002, 4.1",4,2,True,0.15,holds,holds,holds,holds,undecided
1,"Tratamiento, Tabla 1(a)","vdW 2002, 4.1",4,2,True,0.15,holds,holds,holds,holds,holds
2,"Tratamiento, Tabla 1(b)","vdW 2002, 4.1",4,2,True,0.16,holds,holds,holds,holds,holds
3,Progresión por etapas,"vdW 2002, 4.3",5,4,True,0.27,holds,holds,holds,holds,holds
4,Multicepa,"vdW 2002, 4.4",3,2,True,0.12,holds,holds,holds,holds,holds
5,Vector-huésped,"vdW 2002, 4.5",4,2,True,0.08,holds,holds,holds,holds,holds
6,Infección crónica,"CH 2017, Ec. (1)",3,2,True,0.04,holds,holds,holds,holds,holds
7,Proliferación hepática,"CH 2017, Ec. (3)",3,2,True,0.23,holds,holds,holds,holds,undecided
8,Infección aguda con latencia,"CH 2017, Ec. (6)",4,3,True,0.04,holds,holds,holds,holds,fails
9,"Autorreplicación, R0,I","CH 2017, Ec. (7)",3,2,True,0.11,holds,holds,holds,holds,undecided
